# **ZERO-SHOT CLASSIFICATION FOR LLAMA-3.2-3B**


In [2]:
# load test dataset, retrieve this from the .csv file

import pandas as pd

# import warnings
# warnings.filterwarnings('ignore')

test_frame = pd.read_csv('fpb_test.csv')

text_col = 'text'  # text
label_col = 'label' # label
source_col = 'source' # sure why not
unique_labels = test_frame[label_col].unique().tolist()

print(f"loaded {len(test_frame)} rows.")
print(f"labels to test: {unique_labels}")

display(test_frame.head(10))

loaded 727 rows.
labels to test: ['Neutral', 'Fear', 'Optimism', 'Sadness', 'Joy']


,text,label,source,clean_text
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank,the share capital of alma media corporation bu...
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank,the eu commission said earlier it had fined th...
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank,kesko pursues a strategy of healthy focused gr...
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank,down to eurNUM NUM m hNUM NUM NUM august NUM f...
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank,cencorp would focus on the development manufac...
5,"order intake , on the other hand , is expected...",Optimism,FinancialPhraseBank,order intake on the other hand is expected to ...
6,"according to kesko , the company agreed with t...",Optimism,FinancialPhraseBank,according to kesko the company agreed with the...
7,among the scandinavian companies present in st...,Neutral,FinancialPhraseBank,among the scandinavian companies present in st...
8,the terms of the financing were approved by th...,Optimism,FinancialPhraseBank,the terms of the financing were approved by th...
9,finland 's leading metals group outokumpu said...,Optimism,FinancialPhraseBank,finland s leading metals group outokumpu said ...


In [3]:
# log into huggingface

import os
import sys

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")

if google_colab:
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [4]:
!pip install -q transformers accelerate bitsandbytes scikit-learn

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import logging
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

logging.set_verbosity(logging.ERROR)

# configure model
model_id = "meta-llama/Llama-3.2-3B-Instruct" # it has to be the instruct model

# load model and tokeniser
tokenizer = AutoTokenizer.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"

# load in 4-bit to standard colab gpu memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config
)

# Initialize the text generation pipeline once globally
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
    torch_dtype=torch.bfloat16,
    pad_token_id=tokenizer.eos_token_id
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [5]:
def classify(text, candidate_labels):

    # Update prompt to explicitly ask for one of the candidate labels
    # And make sure the system prompt is aligned with the desired single-word output
    prompt_text = f"Classify the sentiment of the following text into one of these categories: {', '.join(candidate_labels)}."
    prompt = [
        {"role": "system", "content": "You are a helpful assistant that classifies text. Respond with only the category name."},
        {"role": "user", "content": f"{prompt_text}\nText: '{text}'"},
    ]

    # Use the globally initialized generator

    generation = generator(
        prompt,
        do_sample=True, # Explicitly set do_sample to True for more diverse responses
        temperature=1.0, # Further increased temperature for more diverse responses
        top_p=0.9, # Slightly reduced top_p to focus sampling on more probable tokens
        max_new_tokens=5
    )

    # Extract the generated text from the full conversation history
    full_conversation = generation[0]['generated_text']
    #print(full_conversation) - debug

    predicted_text = ""
    for message in reversed(full_conversation):
        if message.get('role') == 'assistant':
            predicted_text = message.get('content', '').strip()
            break

    #print(predicted_text) - debug

    # Attempt to find an exact match for the predicted text within the candidate labels
    predicted_label = "Unknown"
    predicted_text_lower = predicted_text.lower()
    for label in candidate_labels:
        if label.lower() == predicted_text_lower:
            predicted_label = label
            break

    return predicted_label

# time to evaluate!
y_true = test_frame[label_col].tolist()
y_pred = [classify(text, unique_labels) for text in test_frame[text_col]]

# print(y_true)
# print(y_pred)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [6]:
# basic metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, labels=unique_labels, zero_division=0)

# Create a DataFrame for per-class F1 scores
per_class_f1_df = pd.DataFrame({
    'Category': unique_labels,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'Support': support
})

print("Per-class F1 Scores:")
display(per_class_f1_df)

# For overall metrics, we can still calculate them (e.g., weighted average)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', labels=unique_labels, zero_division=0)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', labels=unique_labels, zero_division=0)

# export metrics (overall weighted metrics)
results_dict = {
    'Metric': ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'F1 Score (Macro)'] and ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'F1 Score (Macro)'],
    'Value': [accuracy, precision_weighted, recall_weighted, f1_weighted, precision_macro, recall_macro, f1_macro]
}

# create dataframe
results_df = pd.DataFrame(results_dict)
display(results_df)

dashboard_dict = {
    'model': 'Llama 3.2 3B',
    'strategy': 'zero-shot',
    'accuracy': accuracy,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted,
    'f1_fear': per_class_f1_df.at[1, 'F1 Score'],
    'f1_joy': per_class_f1_df.at[4, 'F1 Score'],
    'f1_neutral': per_class_f1_df.at[0, 'F1 Score'],
    'f1_optimism': per_class_f1_df.at[2, 'F1 Score'],
    'f1_sadness': per_class_f1_df.at[3, 'F1 Score']
}

dashboard_df = pd.DataFrame(dashboard_dict, index=[0])

# export to csv
dashboard_df.to_csv('zero_shot_llama_dashboard.csv', index=False)

print("Results saved to 'zero_shot_llama_dashboard.csv'")
display(dashboard_df)

Per-class F1 Scores:


,Category,Precision,Recall,F1 Score,Support
0,Neutral,0.616959,0.976852,0.756272,432
1,Fear,0.090909,0.285714,0.137931,7
2,Optimism,0.500000,0.040816,0.075472,196
3,Sadness,0.800000,0.048193,0.090909,83
4,Joy,0.000000,0.000000,0.000000,9


,Metric,Value
0,Accuracy,0.599725
1,Precision (Weighted),0.593621
2,Recall (Weighted),0.599725
3,F1 Score (Weighted),0.481449
4,Precision (Macro),0.401574
5,Recall (Macro),0.270315
6,F1 Score (Macro),0.212117


Results saved to 'zero_shot_llama_dashboard.csv'


,model,strategy,accuracy,f1_macro,f1_weighted,f1_fear,f1_joy,f1_neutral,f1_optimism,f1_sadness
0,Llama 3.2 3B,zero-shot,0.599725,0.212117,0.481449,0.137931,0.0,0.756272,0.075472,0.090909
